# Lab 13: Free APIs and Integration - The Complete Digital Twin

## Putting it together

This lab turns your own bio into a chatbot persona that remembers the chat. We call it a "digital twin".

The cells below set up `call_groq`. Then you write `build_digital_twin_system_prompt(profile)`. At the end you run a real chat with many turns. The chat keeps adding to a `messages` list, one turn at a time.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


## Exercise

### Goal
Write one function. It turns a dict about a person into a system prompt (the first message in a chat, which tells the model who to be). You use it in the project below to build your own digital twin. Hover over the code in the cell below for the steps and examples.

### What the checks below test
The code cell builds a profile for "Vera" and calls your function. Three `assert` lines check that the name `Vera`, the fact text `FastAPI`, and the boundary text `financial or legal advice` are in your prompt. When every assert passes, the cell prints **All checks passed.**

Also read your printed prompt yourself. Make sure the order and the bullets look right.

In [ ]:
def build_digital_twin_system_prompt(profile: dict) -> str:
    raise NotImplementedError("build_digital_twin_system_prompt() is not implemented yet - replace the TODO with your code")


test_profile = {
    "name": "Vera",
    "role": "a backend engineer learning LLM foundations",
    "tone": "direct and a little dry",
    "facts": ["works primarily in Python and FastAPI", "is building a course on LLM foundations"],
    "boundaries": ["do not give financial or legal advice", "do not make up specific employers or dates"],
}
prompt = build_digital_twin_system_prompt(test_profile)
assert "Vera" in prompt, 'expected ' + repr("Vera") + ' to be in prompt, but it is ' + repr(prompt)
assert "FastAPI" in prompt, 'expected ' + repr("FastAPI") + ' to be in prompt, but it is ' + repr(prompt)
assert "financial or legal advice" in prompt, 'expected ' + repr("financial or legal advice") + ' to be in prompt, but it is ' + repr(prompt)

print("All checks passed.")

## Solution

In [ ]:
def build_digital_twin_system_prompt(profile: dict) -> str:
    lines = [
        f"You are role-playing as {profile['name']}, {profile['role']}.",
        f"Speak in a tone that is {profile['tone']}.",
        "Facts you can draw on about this person:",
    ]
    lines += [f"- {fact}" for fact in profile["facts"]]
    lines.append("Boundaries you must respect:")
    lines += [f"- {b}" for b in profile["boundaries"]]
    return "\n".join(lines)


print(build_digital_twin_system_prompt(test_profile))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Fill in `my_profile` below with your own real bio. Then run a chat with many turns against it. Watch `messages` grow after each turn. That is the whole trick.

In [ ]:
my_profile = {
    "name": "your name",
    "role": "however you'd describe your role in one clause",
    "tone": "a few adjectives for how you talk",
    "facts": [
        "a real fact about you",
        "another real fact about you",
    ],
    "boundaries": [
        "something this digital twin should always refuse to do",
    ],
}

In [ ]:
messages = [{"role": "system", "content": build_digital_twin_system_prompt(my_profile)}]

conversation_turns = [
    "Hi! What are you working on lately?",
    "What's your take on learning by building real projects vs. reading theory?",
]

for user_message in conversation_turns:
    messages.append({"role": "user", "content": user_message})
    reply = call_groq(messages)
    messages.append({"role": "assistant", "content": reply})
    print(f"You: {user_message}")
    print(f"Twin: {reply}\n")

print(f"Final messages list has {len(messages)} turns.")